In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm

In [ ]:
!pip install grad-cam

In [ ]:
IMG_DIR1 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1/"
IMG_DIR2 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2/"
MASK_DIR = "/kaggle/input/datasets/tschandl/ham10000-lesion-segmentations/HAM10000_segmentations_lesion_tschandl/"

CSV_PATH = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"

df = pd.read_csv(CSV_PATH)

df['path'] = df['image_id'].apply(
    lambda x: os.path.join(IMG_DIR1, x+'.jpg') 
    if os.path.exists(os.path.join(IMG_DIR1, x+'.jpg')) 
    else os.path.join(IMG_DIR2, x+'.jpg')
)

df['mask_path'] = df['image_id'].apply(
    lambda x: os.path.join(MASK_DIR, x+'_segmentation.png')
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['dx'])

NUM_CLASSES = df['label'].nunique()

In [ ]:
train_aug = A.Compose([
    A.Resize(256,256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=40, p=0.7),
    A.RandomBrightnessContrast(p=0.5),
    A.CoarseDropout(p=0.5),
    A.Normalize(),
    ToTensorV2()
], additional_targets={'mask': 'mask'})  # keep this

In [ ]:
class SkinDataset(Dataset):
    def __init__(self, df, aug=None):
        self.df = df
        self.aug = aug
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img = cv2.imread(row['path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(row['mask_path'], 0)
        
        if mask is None:
            mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)

        if self.aug:
            augmented = self.aug(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']   # tensor (uint8!)
        else:
            mask = torch.tensor(mask)

        
        mask = (mask > 0).float()

        # Ensure [1, H, W]
        if len(mask.shape) == 2:
            mask = mask.unsqueeze(0)

        label = torch.tensor(row['label']).long()
        
        return img, mask, label

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.encoder = timm.create_model(
            'resnet34',
            pretrained=True,
            features_only=True
        )
        
        # Simple decoder (fixed)
        self.up1 = nn.ConvTranspose2d(512,256,2,2)
        self.up2 = nn.ConvTranspose2d(256,128,2,2)
        self.up3 = nn.ConvTranspose2d(128,64,2,2)
        self.up4 = nn.ConvTranspose2d(64,32,2,2)
        self.up5 = nn.ConvTranspose2d(32,16,2,2)  

        self.final = nn.Conv2d(16,1,1)

    def forward(self,x):
        feats = self.encoder(x)
        x = feats[-1]  # 8x8 feature map

        x = self.up1(x)  # 16x16
        x = self.up2(x)  # 32x32
        x = self.up3(x)  # 64x64
        x = self.up4(x)  # 128x128
        x = self.up5(x)  #  256x256

        return self.final(x)


def dice_loss(pred, target):
    pred = torch.sigmoid(pred)

    # flatten per sample
    pred = pred.view(pred.size(0), -1)
    target = target.view(target.size(0), -1)

    smooth = 1e-6

    intersection = (pred * target).sum(dim=1)
    dice = (2 * intersection + smooth) / (pred.sum(dim=1) + target.sum(dim=1) + smooth)

    return 1 - dice.mean()

def seg_loss(pred, target):
    return 0.5 * bce(pred, target) + 0.5 * dice_loss(pred, target)


device = 'cuda'

seg_model = UNet().to(device)
optimizer = torch.optim.Adam(seg_model.parameters(), lr=1e-4)

train_ds = SkinDataset(df, train_aug)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

for epoch in range(10):
    seg_model.train()
    total_loss = 0
    
    for img, mask, _ in train_loader:
        img = img.to(device)
        mask = mask.to(device)

        pred = seg_model(img)

        pred = F.interpolate(pred, size=mask.shape[-2:], mode='bilinear', align_corners=False)

        loss = seg_loss(pred, mask)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch} Loss: {total_loss/len(train_loader)}")

In [ ]:
torch.save(seg_model.state_dict(), "unet_segmentation.pth")

In [ ]:
def apply_mask(img, mask):
    mask = (mask > 0.5).float()
    mask = mask.to(img.device)

    if mask.shape != img.shape:
        mask = mask.expand_as(img)

    return img * mask

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=NUM_CLASSES)
    
    def forward(self,x):
        return self.model(x)

In [ ]:
clf_model = Classifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(clf_model.parameters(), lr=1e-4)

for epoch in range(10):
    clf_model.train()
    total_loss = 0
    
    for img, mask, label in train_loader:
        img = img.to(device)
        mask = mask.to(device)
        label = label.to(device)
        
        with torch.no_grad():
            seg = torch.sigmoid(seg_model(img))
            img = apply_mask(img, seg)
        
        pred = clf_model(img)
        loss = criterion(pred, label)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch} Loss: {total_loss/len(train_loader)}")

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# Set model to eval
clf_model.eval()

# ✅ FIXED TARGET LAYER
target_layer = clf_model.model.conv_head

# Create CAM
cam = GradCAM(model=clf_model, target_layers=[target_layer])

# Get sample
img, mask, label = train_ds[0]
input_tensor = img.unsqueeze(0).to(device)

# Generate CAM
grayscale_cam = cam(input_tensor=input_tensor)[0]

# -------- UNNORMALIZE IMAGE --------
img_np = img.permute(1, 2, 0).cpu().numpy()

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

img_np = (img_np * std) + mean
img_np = np.clip(img_np, 0, 1).astype(np.float32)

# -------- OVERLAY --------
visualization = show_cam_on_image(
    img_np,
    grayscale_cam,
    use_rgb=True
)

# Display
plt.figure(figsize=(6,6))
plt.imshow(visualization)
plt.title(f"Grad-CAM | Label: {label}")
plt.axis('off')
plt.show()